# Capítulo 6 · Descrevendo duas variáveis: caderno complementar
*Ciência de Dados e Inteligência Artificial em Saúde, Volume 1*

> **Este caderno acompanha o capítulo do livro; não o reproduz.** Definições, deduções e explicações estão no livro. Aqui você executa os cálculos do caso e confere os números. Todos os dados são **sintéticos** e os valores têm apenas finalidade didática.

A **Parte A** percorre o caso do capítulo, um mês do registro de triagem de um pronto-socorro, pelas perguntas da diretoria; aparecem só as seções com cálculo (6.2, 6.3 e 6.5 a 6.11). O livro desenha as figuras; o caderno imprime os mesmos números. A **Parte B** traz a resolução escrita de todos os exercícios do capítulo (1 a 14, e a nota técnica sobre as coordenadas dos mapas perceptuais; os enunciados estão no livro) e, nos exercícios cuja resolução precisa de cálculo (2, 4, 7, 9, 11, 13 e 14), o código e o resultado. O código é Python 3 com pandas. Execute as células em ordem.

## Parte A. O caso

**Dados.** `data/triage_month.csv`: 1.263 linhas, uma por atendimento de emergência em março de 2026, e 13 colunas: `visit_id`, `date`, `arrived_at`, `seen_at`, `arrival_route` (1 demanda espontânea, 2 ambulância, 3 transferência), `sex`, `age_years`, `triage_level` (1 mais urgente a 5 menos urgente), `pain`, `temp_c`, `heart_rate`, `prior_visits` e `admitted`. Cada célula traz o número da seção do livro a que pertence.

### 6.2 A tabela de contingência

*Como a internação difere entre as vias de chegada?*

In [1]:
# 1. Ler o registro de triagem de março: uma linha por atendimento
import pandas as pd

df = pd.read_csv("data/triage_month.csv")

# 2. A tabela de contingência de via de chegada e internação, com os totais (margens)
table = pd.crosstab(df["arrival_route"], df["admitted"], margins=True)
print(table)

# 3. Perfis das linhas: a porcentagem de cada linha (via de chegada) que cai em cada coluna
rows = pd.crosstab(df["arrival_route"], df["admitted"], normalize="index")
print((100 * rows).round(1))

admitted        no  yes   All
arrival_route                
1              722  199   921
2              152   81   233
3               74   35   109
All            948  315  1263
admitted         no   yes
arrival_route            
1              78.4  21.6
2              65.2  34.8
3              67.9  32.1


### 6.3 Gráficos para duas variáveis qualitativas

*Como as duas variáveis podem ser mostradas juntas?*

In [2]:
# 1. Contagens de cada via de chegada por internação, internados primeiro (o livro desenha as barras agrupadas)
counts = pd.crosstab(df["arrival_route"], df["admitted"]).reindex(columns=["yes", "no"])
print(counts)

# 2. Perfis das linhas em %, que levam cada barra empilhada a 100 (o livro desenha as barras empilhadas)
profiles = 100 * pd.crosstab(df["arrival_route"], df["admitted"], normalize="index")
print(profiles.reindex(columns=["yes", "no"]).round(1))

admitted       yes   no
arrival_route          
1              199  722
2               81  152
3               35   74
admitted        yes    no
arrival_route            
1              21.6  78.4
2              34.8  65.2
3              32.1  67.9


### 6.5 Phi e V de Cramér

*Quão forte é a associação, em uma escala de 0 a 1?*

In [3]:
# 1. Contagens esperadas se via de chegada e internação fossem independentes, e a estatística qui-quadrado (sem correção de continuidade)
from scipy.stats import chi2_contingency

observed = pd.crosstab(df["arrival_route"], df["admitted"])
chi2, _, _, expected = chi2_contingency(observed, correction=False)
print("Contagens esperadas (colunas: alta, internado):")
print(expected.round(1))

# 2. Qui-quadrado e V de Cramér para a via de chegada
n = len(df)
print("Via de chegada: qui-quadrado", round(chi2, 2), "; V de Cramér", round((chi2 / n) ** 0.5, 3))

# 3. O mesmo para o nível de triagem e a internação
levels = pd.crosstab(df["triage_level"], df["admitted"])
chi2_levels = chi2_contingency(levels, correction=False)[0]
print("Nível de triagem: qui-quadrado", round(chi2_levels, 1), "; V de Cramér", round((chi2_levels / n) ** 0.5, 3))

Contagens esperadas (colunas: alta, internado):
[[691.3 229.7]
 [174.9  58.1]
 [ 81.8  27.2]]
Via de chegada: qui-quadrado 20.47 ; V de Cramér 0.127
Nível de triagem: qui-quadrado 388.2 ; V de Cramér 0.554


### 6.6 Mapas perceptuais e análise de correspondência

*Como uma tabela com muitas categorias pode ser desenhada como um mapa?*

In [4]:
# 1. Perfis das linhas da via de chegada por nível de triagem e a participação de cada nível de triagem no registro inteiro
route_level = pd.crosstab(df["arrival_route"], df["triage_level"])
row_profiles = route_level.div(route_level.sum(axis=1), axis=0)
column_share = route_level.sum(axis=0) / len(df)

# 2. Distância qui-quadrado entre os perfis dos atendimentos de demanda espontânea (1) e de ambulância (2) e a inércia total (qui-quadrado dividido por n)
difference = row_profiles.loc[1] - row_profiles.loc[2]
distance = ((difference ** 2 / column_share).sum()) ** 0.5
chi2_map = chi2_contingency(route_level, correction=False)[0]
print("Distância entre os perfis de demanda espontânea e ambulância:", round(distance, 3), "; inércia total:", round(chi2_map / len(df), 5))

Distância entre os perfis de demanda espontânea e ambulância: 0.871 ; inércia total: 0.11867


In [5]:
# 3. Análise de correspondência à mão: resíduos padronizados e sua decomposição em valores singulares
import numpy as np

p = route_level.to_numpy(dtype=float) / len(df)
r = p.sum(axis=1)
c = p.sum(axis=0)
expected_share = r[:, None] * c[None, :]
s = (p - expected_share) / np.sqrt(expected_share)
u, singular, vt = np.linalg.svd(s, full_matrices=False)
row_xy = u[:, :2] * singular[:2] / np.sqrt(r[:, None])
column_xy = vt.T[:, :2] * singular[:2] / np.sqrt(c[:, None])

# 4. A parte da inércia carregada por cada eixo e a distância entre demanda espontânea e ambulância no mapa
axis_share = 100 * singular[:2] ** 2 / (singular ** 2).sum()
print("Inércia carregada pelo primeiro e pelo segundo eixo (%):", axis_share.round(1).tolist())
print("Distância entre demanda espontânea e ambulância no mapa:", round(np.sqrt(((row_xy[0] - row_xy[1]) ** 2).sum()), 3))

Inércia carregada pelo primeiro e pelo segundo eixo (%): [97.4, 2.6]
Distância entre demanda espontânea e ambulância no mapa: 0.871


### 6.7 Uma variável quantitativa descrita por grupo

*Como idade e frequência cardíaca diferem entre atendimentos com internação e com alta?*

In [6]:
# 1. Idade por situação de internação: contagem, média, desvio padrão e mediana (o livro desenha os boxplots)
by_group = df.groupby("admitted")
print(by_group["age_years"].agg(["count", "mean", "std", "median"]).round(1))

# 2. Frequência cardíaca: os três quartis em cada grupo
print(by_group["heart_rate"].quantile([0.25, 0.5, 0.75]).round(1))

          count  mean   std  median
admitted                           
no          948  51.7  17.9    52.0
yes         315  55.2  19.1    55.0
admitted      
no        0.25     84.0
          0.50     93.0
          0.75    102.0
yes       0.25     87.5
          0.50     97.0
          0.75    107.5
Name: heart_rate, dtype: float64


### 6.8 O diagrama de dispersão

*Como frequência cardíaca e temperatura se movem juntas?*

In [7]:
# 1. Frequência cardíaca e temperatura em classes (o livro desenha o diagrama de dispersão): a contagem de cada combinação
hr_class = pd.cut(df["heart_rate"], [-np.inf, 80, 100, 120, np.inf], right=False)
temp_class = pd.cut(df["temp_c"], [-np.inf, 37, 38, 39, np.inf], right=False)
joint = pd.crosstab(hr_class, temp_class)
print("Contagens por classe de frequência cardíaca (linhas) e de temperatura (colunas):", joint.to_numpy().tolist())

Contagens por classe de frequência cardíaca (linhas) e de temperatura (colunas): [[123, 76, 0, 1], [321, 240, 33, 44], [133, 91, 77, 87], [3, 3, 10, 21]]


### 6.9 Covariância

*Frequência cardíaca e temperatura sobem juntas, e quanto?*

In [8]:
# 1. Covariância amostral de frequência cardíaca e temperatura: os oito primeiros atendimentos (cálculo manual) e o mês inteiro
first = df.head(8)
print("Covariância, oito primeiros atendimentos:", round(first["heart_rate"].cov(first["temp_c"]), 2))
print("Covariância, mês inteiro:", round(df["heart_rate"].cov(df["temp_c"]), 2))

Covariância, oito primeiros atendimentos: 4.61
Covariância, mês inteiro: 6.25


### 6.10 Coeficiente de correlação de Pearson

*Quão forte é a associação linear, livre das unidades?*

In [9]:
# 1. Correlação de Pearson de frequência cardíaca e temperatura: os oito primeiros atendimentos e o mês inteiro
print("Correlação, oito primeiros atendimentos:", round(first["heart_rate"].corr(first["temp_c"]), 2))
print("Correlação, mês inteiro:", round(df["heart_rate"].corr(df["temp_c"]), 3))

# 2. A matriz de correlações de idade, frequência cardíaca e temperatura
print(df[["age_years", "heart_rate", "temp_c"]].corr().round(2))

Correlação, oito primeiros atendimentos: 0.18
Correlação, mês inteiro: 0.427
            age_years  heart_rate  temp_c
age_years        1.00       -0.00   -0.03
heart_rate      -0.00        1.00    0.43
temp_c          -0.03        0.43    1.00


### 6.11 Correlação de postos para variáveis ordenadas

*Como duas variáveis ordenadas se relacionam?*

In [10]:
# 1. Categorias de dor como códigos na ordem da escala (só para ordená-las; não se afirma que os degraus sejam iguais)
pain_code = df["pain"].map({"none": 0, "mild": 1, "moderate": 2, "severe": 3})

# 2. Correlação de Spearman do nível de triagem com a dor: os oito primeiros atendimentos e o mês inteiro
print("Spearman, nível de triagem e dor, oito primeiros atendimentos:", round(first["triage_level"].corr(pain_code.head(8), method="spearman"), 2))
print("Spearman, nível de triagem e dor, mês inteiro:", round(df["triage_level"].corr(pain_code, method="spearman"), 2))

# 3. Correlação de Spearman do nível de triagem com a frequência cardíaca
print("Spearman, nível de triagem e frequência cardíaca:", round(df["triage_level"].corr(df["heart_rate"], method="spearman"), 2))

Spearman, nível de triagem e dor, oito primeiros atendimentos: -0.47
Spearman, nível de triagem e dor, mês inteiro: -0.25
Spearman, nível de triagem e frequência cardíaca: -0.27


## Parte B. Soluções dos exercícios

### 6.14 Exercício 1. Sexo e internação

| Sexo | Internados | Com alta | Total |
|---|---|---|---|
| Feminino | 150 | 504 | 654 |
| Masculino | 165 | 444 | 609 |
| Total | 315 | 948 | 1.263 |



Para os homens internados: porcentagem na linha $165/609=27{,}1\%$, a proporção
de homens que foram internados; porcentagem na coluna $165/315=52{,}4\%$, a
proporção de pacientes internados que eram homens; porcentagem conjunta
$165/1.263=13{,}1\%$, a proporção de todos os atendimentos que foram homens
internados. A porcentagem na linha responde à pergunta, em comparação com
$150/654=22{,}9\%$ entre as mulheres: os homens foram internados um pouco mais
vezes.

### 6.14 Exercício 2. Dor e internação

Os perfis das linhas mostram internação em 16,7% dos 293 atendimentos sem dor,
23,1% dos 432 com dor leve, 29,8% dos 372 com dor moderada e 33,1% dos 166 com
dor intensa: a proporção de internados aumenta ao longo da escala de dor.

In [11]:
# 1. Perfis das linhas (%) da internação por dor, na ordem da escala
order = ["none", "mild", "moderate", "severe"]
pain = pd.Categorical(df["pain"], categories=order, ordered=True)
print((100 * pd.crosstab(pain, df["admitted"], normalize="index")).round(1))

admitted    no   yes
row_0               
none      83.3  16.7
mild      76.9  23.1
moderate  70.2  29.8
severe    66.9  33.1


### 6.14 Exercício 3. Contagens esperadas, qui-quadrado e phi

Contagens esperadas: feminino internado $654\times315/1.263=163{,}1$, feminino
com alta $490{,}9$, masculino internado $151{,}9$, masculino com alta $457{,}1$;
as linhas somam 654 e 609 e as colunas 315 e 948. Toda contagem observada difere
da esperada em 13,1, de modo que
$\chi^2=171{,}6\,(1/163{,}1+1/490{,}9+1/151{,}9+1/457{,}1)=1{,}05+0{,}35+1{,}13+0{,}38=2{,}91$
e $\phi=\sqrt{2{,}91/1.263}=0{,}048$. Em comparação com $0{,}127$ para a via de
chegada, a associação da internação com o sexo é ainda menor; ambas são fracas
ao lado de $0{,}554$ para o nível de triagem.

### 6.14 Exercício 4. V de Cramér para dor e para nível de triagem

As duas tabelas têm duas colunas, de modo que $\min(r-1,c-1)=1$ e $V=\phi$
numericamente; $V$ usa uma escala de 0 a 1, mas as proporções das categorias e
suas definições ainda afetam seu valor. Para verificar o limite superior,
escreva $R_i$ e $C_j$ para os totais positivos das linhas e das colunas.
Expandir as diferenças ao quadrado dá

$$ \frac{\chi^2}{n}=\sum_{i,j}\frac{O_{ij}^2}{R_iC_j}-1. $$

Como $O_{ij}\leq C_j$, a soma é no máximo
$\sum_{i,j}O_{ij}/R_i=r$. Usar $O_{ij}\leq R_i$ em vez disso a limita por $c$.
Assim, $0\leq\chi^2/n\leq\min(r-1,c-1)$, e a divisão por esse limite superior
seguida de uma raiz quadrada dá $0\leq V\leq1$.
A internação está associada à dor, $V=0{,}13$,
tão fracamente quanto à via de chegada, e ao nível de triagem de modo muito
mais forte, $V=0{,}55$.

In [12]:
# 1. Qui-quadrado e V de Cramér da dor e do nível de triagem contra a internação
n = len(df)
for col in ["pain", "triage_level"]:
    chi2 = chi2_contingency(pd.crosstab(df[col], df["admitted"]), correction=False)[0]
    print(col + ": qui-quadrado", round(chi2, 1), "; V de Cramér", round((chi2 / n) ** 0.5, 3))

pain: qui-quadrado 22.0 ; V de Cramér 0.132
triage_level: qui-quadrado 388.2 ; V de Cramér 0.554


### 6.14 Exercício 5. Spearman à mão

Os códigos de dor 3, 2, 2, 1, 0, 1 recebem os postos 6, 4,5, 4,5, 2,5, 1, 2,5
(os dois 2 compartilham os postos 4 e 5; os dois 1 compartilham 2 e 3). Os
níveis de triagem 1, 2, 3, 3, 5, 4 recebem os postos 1, 2, 3,5, 3,5, 6, 5. O
coeficiente de Pearson das duas séries de postos é $\rho=-0{,}93$: quanto mais
intensa a dor, mais urgente (menor) o nível. As diferenças $d_i$ são 5, 2,5, 1,
$-1$, $-5$, $-2{,}5$, com $\sum d_i^2=64{,}5$, e o atalho dá
$1-6(64{,}5)/(6\times35)=-0{,}84$, não $-0{,}93$; o atalho pressupõe ausência
de empates e está errado aqui, onde as duas séries os têm.

### 6.14 Exercício 6. Diferenças entre grupos e sobreposição individual

As médias resumem os grupos, enquanto as caixas sobrepostas mostram que muitas
idades ocorrem nos dois. Uma idade mais avançada não é uma regra que determina
a internação. A espera mais curta entre os atendimentos com internação é uma
associação: o nível de triagem influencia tanto a regra de espera quanto a
internação neste exemplo sintético. Entender por que os grupos diferem exige
informação sobre esse processo e sobre características relevantes dos pacientes;
as duas medianas isoladas não estabelecem um efeito da internação sobre o tempo
de espera.

### 6.14 Exercício 7. Temperatura e dor segundo o desfecho da internação

Temperatura: 37,47 (1,15) entre os internados, 37,25 (1,03) entre os que
tiveram alta, em graus Celsius. Os histogramas mostram um grupo de temperatura
mais baixa e um grupo febril menor; uma única média e um único DP ocultam essa
estrutura. Dor: nenhuma 49 (15,6%) contra 244 (25,7%), leve 100 (31,7%)
contra 332 (35,0%), moderada 111 (35,2%) contra 261 (27,5%), intensa 55
(17,5%) contra 111 (11,7%).

In [13]:
# 1. Temperatura por situação de internação: média e desvio padrão
temp = df.groupby("admitted")["temp_c"].agg(["mean", "std"]).round(2)
print(temp)

# 2. Dor por situação de internação: contagens e porcentagens de cada coluna
counts = pd.crosstab(pain, df["admitted"])
share = (100 * pd.crosstab(pain, df["admitted"], normalize="columns")).round(1)
for level in order:
    print("Dor", level + ": internados", counts.loc[level, "yes"], "(" + str(share.loc[level, "yes"]) + "%); alta", counts.loc[level, "no"], "(" + str(share.loc[level, "no"]) + "%)")

           mean   std
admitted             
no        37.25  1.03
yes       37.47  1.15
Dor none: internados 49 (15.6%); alta 244 (25.7%)
Dor mild: internados 100 (31.7%); alta 332 (35.0%)
Dor moderate: internados 111 (35.2%); alta 261 (27.5%)
Dor severe: internados 55 (17.5%); alta 111 (11.7%)


### 6.14 Exercício 8. Covariância e correlação à mão

Médias $\bar{x}=57{,}4$ anos e $\bar{y}=55{,}0$ minutos. Desvios da idade:
$-26{,}4$, 23,6, 13,6, 11,6, $-22{,}4$; do tempo de espera: 92, $-12$, $-12$,
$-50$, $-18$. Produtos: $-2.428{,}8$, $-283{,}2$, $-163{,}2$, $-580{,}0$, 403,2;
soma $-3.052$; covariância $-3.052/4=-763$ ano-minutos. Desvios padrão
$s_x=22{,}78$ anos e $s_y=53{,}8$ minutos, de modo que
$r=-763/(22{,}78\times53{,}8)=-0{,}62$, sem unidade. Nestes cinco atendimentos,
os pacientes mais velhos esperaram menos; cinco atendimentos não descrevem nada
além de si mesmos.

### 6.14 Exercício 9. Matriz de correlação e nível de triagem contra tempo de espera

A matriz mostra pouca associação linear da idade com as outras três, frequência
cardíaca e temperatura em $0{,}43$, e o tempo de espera fracamente relacionado
de modo negativo à frequência cardíaca ($-0{,}22$) e à temperatura ($-0{,}14$).
O coeficiente de Spearman entre o nível de triagem e o tempo de espera é
$0{,}82$: quanto menos urgente o nível, mais longa a espera, de modo quase
monótono. Ele é grande porque a regra do serviço o produz: o nível de triagem é
o critério pelo qual os pacientes são chamados, de modo que a relação está
embutida no processo, e não descoberta nos pacientes.

In [14]:
# 1. Tempo de espera em minutos e a matriz de correlações de idade, frequência cardíaca, temperatura e tempo de espera
df["wait"] = (pd.to_datetime(df["seen_at"]) - pd.to_datetime(df["arrived_at"])).dt.total_seconds() / 60
print(df[["age_years", "heart_rate", "temp_c", "wait"]].corr().round(2))

# 2. Correlação de Spearman do nível de triagem com o tempo de espera
print("Spearman, nível de triagem e tempo de espera:", round(df["triage_level"].corr(df["wait"], method="spearman"), 2))

            age_years  heart_rate  temp_c  wait
age_years        1.00       -0.00   -0.03 -0.00
heart_rate      -0.00        1.00    0.43 -0.22
temp_c          -0.03        0.43    1.00 -0.14
wait            -0.00       -0.22   -0.14  1.00
Spearman, nível de triagem e tempo de espera: 0.82


### 6.14 Exercício 10. O que o coeficiente não mostra

(a) Concordância: dois oxímetros podem ter um desvio sistemático além do ruído
de medida e ainda assim dar $r=0{,}97$; são necessárias as diferenças entre as
leituras pareadas (Seção 6.12, “Não é concordância”).
(b) Uma relação entre indivíduos: a correlação é entre médias hospitalares e não
diz nada sobre se um paciente mais velho tem maior probabilidade de ser
internado (“Não é uma afirmação sobre indivíduos”). (c) A força da relação em
geral: pacientes de terapia intensiva constituem uma faixa restrita de lactato e
de risco, o que pode alterar $r$ (“Não é independente da faixa estudada”). (d)
Causação: as duas medidas sobem e descem juntas porque ambas refletem a
glicemia; $r$ não pode dizer que uma produz a outra (“Não é causação”). (e) Um
efeito causal: o porte e a atividade do hospital podem explicar tanto uma maior
capacidade de estacionamento quanto mais óbitos. A correlação observada não dá
base para esperar menos óbitos após a remoção de vagas de estacionamento (“Não
é causação”). Comparações de mortalidade também exigem denominadores apropriados
e atenção aos pacientes tratados; contagens anuais isoladas não medem a
qualidade do cuidado.

### 6.14 Exercício 11. Fahrenheit

A covariância é multiplicada por 1,8, de 6,25 para 11,25, porque cada desvio de
temperatura o é; a correlação permanece inalterada em 0,427. A covariância
mostra que carrega as unidades de ambas as variáveis; a correlação ilustra a
parte (a) da Proposição 6.2, a invariância sob uma reescala positiva
e uma translação.

In [15]:
# 1. Temperatura em Fahrenheit e a covariância e a correlação da frequência cardíaca com cada unidade
temp_f = 1.8 * df["temp_c"] + 32
print("Covariância, Celsius:", round(df["heart_rate"].cov(df["temp_c"]), 2), "; Fahrenheit:", round(df["heart_rate"].cov(temp_f), 2))
print("Correlação, Celsius:", round(df["heart_rate"].corr(df["temp_c"]), 3), "; Fahrenheit:", round(df["heart_rate"].corr(temp_f), 3))

Covariância, Celsius: 6.25 ; Fahrenheit: 11.25
Correlação, Celsius: 0.427 ; Fahrenheit: 0.427


### 6.14 Exercício 12. Escolhendo a ferramenta a partir das escalas

(a) Ambas nominais: tabela de contingência com perfis das linhas, barras
agrupadas ou empilhadas, $\phi$ ou $V$. (b) Ambas ordinais: $\rho$ de Spearman
sobre os postos, já que a ordem é a informação; uma tabela de contingência pode
exibir as contagens. (c) Nominal e quantitativa: mediana [$Q_1$, $Q_3$] do tempo
de permanência por enfermaria, se as distribuições dentro das enfermarias forem
assimétricas, e boxplots lado a lado. (d) Ambas quantitativas: diagrama de
dispersão e, em seguida, o $r$ de Pearson, se o gráfico mostrar que um resumo
linear é apropriado. (e) Um desfecho binário contra uma variável nominal de doze
níveis: uma tabela de contingência $12\times2$ com a proporção de reinternação de
cada enfermaria como perfil da linha, um gráfico de barras dessas proporções com
o $n$ de cada enfermaria e o $V$ de Cramér para a associação global; os perfis, e
não o coeficiente, são aquilo sobre o qual o relatório gerencial atua.

### 6.14 Exercício 13. Conferindo um mapa perceptual

Os níveis 1 e 2 respondem por $27/109=24{,}8\%$ das transferências e por
$86/921=9{,}3\%$ dos atendimentos por demanda espontânea. Para o nível 1, a
contribuição para a distância ao quadrado entre eles é

$$ \frac{(4/109-12/921)^2}{44/1.263}\approx0{,}01608. $$

O bloco a seguir recalcula os dois perfis completos e sua distância a partir do
registro de atendimentos. Ele pode ser executado independentemente do código de
coordenadas do capítulo.

As coordenadas plotadas dão
$\sqrt{(0{,}20679+0{,}19531)^2+(0{,}17624+0{,}01122)^2}\approx0{,}444$,
em concordância com o cálculo pela tabela. O primeiro eixo retém 97,4% da
inércia, não 97,4% dos desfechos dos pacientes nem das previsões corretas. A
figura descreve este registro; avaliar a incerteza amostral exige uma análise
inferencial separada.

In [16]:
# 1. Distância entre os perfis dos atendimentos de transferência (3) e de demanda espontânea (1) por nível de triagem
route_level = pd.crosstab(df["arrival_route"], df["triage_level"])
profiles = route_level.div(route_level.sum(axis=1), axis=0)
column_share = route_level.sum(axis=0) / len(df)
difference = profiles.loc[3] - profiles.loc[1]
print("Distância entre os perfis de transferência e demanda espontânea:", round(((difference ** 2 / column_share).sum()) ** 0.5, 3))

Distância entre os perfis de transferência e demanda espontânea: 0.444


### 6.14 Exercício 14. Classes conjuntas e seus denominadores

Há um atendimento com temperatura de pelo menos 38,0 ° C entre os 200
atendimentos com frequência cardíaca abaixo de 80 batimentos/min. As outras três
classes têm 77 de 638, 164 de 388 e 31 de 37. Suas proporções são, portanto,
0,5%, 12,1%, 42,3% e 83,8%. O código a seguir reconstrói as contagens e as
quatro proporções condicionais.

A maior contagem de atendimentos no ponto de corte de temperatura ou acima dele
está na classe de 100-119 batimentos/min; a maior proporção dentro da classe
está na classe de 120 ou mais. Contagens e perfis respondem a perguntas
diferentes. O diagrama de dispersão retém as medidas pareadas reais, incluindo
sua variação dentro de cada classe. O agrupamento substitui cada medida por um
intervalo, e o padrão aparente pode mudar conforme os limites escolhidos.

In [17]:
# 1. Classes conjuntas de frequência cardíaca e temperatura
hr_class = pd.cut(df["heart_rate"], [-np.inf, 80, 100, 120, np.inf], right=False)
temp_class = pd.cut(df["temp_c"], [-np.inf, 37, 38, 39, np.inf], right=False)
joint = pd.crosstab(hr_class, temp_class)
print("Contagens:", joint.to_numpy().tolist())

# 2. A parte de cada classe de frequência cardíaca com temperatura de 38 ou mais (%)
print("Parte com temperatura de 38 ou mais, por classe de frequência cardíaca (%):", (100 * joint.iloc[:, 2:].sum(axis=1) / joint.sum(axis=1)).round(1).tolist())

Contagens: [[123, 76, 0, 1], [321, 240, 33, 44], [133, 91, 77, 87], [3, 3, 10, 21]]
Parte com temperatura de 38 ou mais, por classe de frequência cardíaca (%): [0.5, 12.1, 42.3, 83.8]


### 6.14 Nota técnica: coordenadas dos mapas perceptuais

Esta nota fornece a derivação das coordenadas usada na
Seção 6.6. Uma matriz é um arranjo retangular de números;
um vetor é uma única lista ordenada. O sobrescrito $T$ troca linhas por colunas,
e uma matriz diagonal tem entradas nulas, exceto na diagonal. O produto de duas
matrizes forma cada entrada somando os produtos pareados de uma linha e uma
coluna. A matriz identidade, escrita $I$, tem uns na diagonal e deixa um vetor
inalterado quando multiplicada por ele.

Seja $P=O/n$ a matriz das proporções conjuntas, com margens positivas de linha e
de coluna $r$ e $c$. Escreva $D_r$ e $D_c$ para as matrizes diagonais que contêm
essas margens. A tabela transformada é

$$ S=D_r^{-1/2}(P-rc^T)D_c^{-1/2},
 \qquad S_{ij}=\frac{p_{ij}-r_ic_j}{\sqrt{r_ic_j}}. $$

Aqui $D_r^{-1/2}$ tem entradas $1/\sqrt{r_i}$ na diagonal, e de modo semelhante
para as colunas. A substituição direta dá

$$ \sum_{i,j}S_{ij}^2=\frac{\chi^2}{n}. $$


Uma decomposição em valores singulares escreve $S=U\Sigma V^T$, usando colunas de
comprimento unitário e mutuamente perpendiculares em $U$ e $V$. As entradas
diagonais positivas $\sigma_1\geq\sigma_2\geq\cdots$ de $\Sigma$ dão os tamanhos
desses componentes. Em símbolos, $U^TU=V^TV=I$. Esta é a decomposição numérica
retornada por `numpy.linalg.svd`; a construção das coordenadas abaixo usa
suas propriedades definidoras.

As coordenadas principais das linhas e das colunas são

$$ F=D_r^{-1/2}U\Sigma,\qquad
 G=D_c^{-1/2}V\Sigma. $$

Para verificar o que uma distância entre linhas representa, tome
$Z=D_r^{-1/2}S$. Suas entradas são
$Z_{ij}=(q_{ij}-c_j)/\sqrt{c_j}$ e $Z=FV^T$.
A distância ao quadrado entre duas linhas de $Z$ é, portanto, exatamente
$\sum_j(q_{ij}-q_{i'j})^2/c_j$.
Como $V^TV=I$, a mesma distância ao quadrado é igual a

$$ (F_i-F_{i'})V^TV(F_i-F_{i'})^T
 =\sum_k(F_{ik}-F_{i'k})^2. $$

Assim, quando todos os componentes não nulos são retidos, as distâncias
plotadas comuns entre pontos de linhas são iguais às distâncias qui-quadrado
entre perfis de linhas. Trocar linhas e colunas prova a afirmação correspondente
para $G$. Não há identidade comparável para a distância entre um ponto de linha
e um ponto de coluna.

A contribuição do componente $k$ para a soma dos quadrados das entradas de $S$ é
$\sigma_k^2$: expanda $S_{ij}=\sum_k\sigma_kU_{ik}V_{jk}$, eleve ao quadrado e
some; todos os termos cruzados se anulam porque colunas distintas são
perpendiculares. Logo, $\sum_k\sigma_k^2=\chi^2/n$. Manter os dois primeiros
componentes retém $(\sigma_1^2+\sigma_2^2)/\sum_k\sigma_k^2$ da inércia.
Para qualquer direção unitária $v$, sua soma de quadrados capturada é
$\|Sv\|^2=\sum_k\sigma_k^2(V_k^Tv)^2\leq\sigma_1^2$,
em que $\|\cdot\|^2$ significa a soma dos quadrados das entradas. Os pesos
$(V_k^Tv)^2$ somam no máximo 1; a igualdade no máximo é atingida por
$v=V_1$. Repetir no espaço perpendicular dá a segunda direção.
Isso estabelece a ordenação dos eixos usada no mapa.

Valem tanto $S\sqrt{c}=0$ quanto $\sqrt{r}^{\,T}S=0$, porque as margens
observadas e esperadas coincidem. Consequentemente, o número de componentes não
nulos é no máximo $\min(R-1,C-1)$ para uma tabela $R\times C$, e as médias
ponderadas das coordenadas são nulas. A origem é o perfil global. A tabela
$3\times5$ do capítulo precisa de apenas dois componentes. Em uma tabela maior,
os componentes omitidos subtraem diferenças de coordenadas ao quadrado, não
negativas, de uma distância plotada; pontos projetados próximos podem ocultar
diferenças nas dimensões omitidas. Inverter o sinal de um eixo preserva todas as
distâncias e a interpretação.

---
*Dados sintéticos (`generate_data.py`, semente 60). Valores apenas didáticos; nada aqui é estimativa para um departamento real.*